In [10]:
# ParlaSpeech V2 metadata inspection (daily resolution + speaker/location coverage)

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from datetime import datetime
from tqdm import tqdm

# --- CONFIG ---
csv_folder = "/home/tom/projects/corpora/data/parlspeech"  # folder with converted CSVs
output_dir = "/home/tom/projects/corpora/data/processed/"
os.makedirs(output_dir, exist_ok=True)

# --- Accumulator ---
meta_output = []

# --- Helper function ---
def safe_date(val):
    try:
        return pd.to_datetime(val).date()
    except:
        return None

csv_files = glob(os.path.join(csv_folder, "*.csv"))

for file_path in tqdm(csv_files, desc="Processing CSV files"):
    try:
        df = pd.read_csv(file_path)
        df = df.dropna(subset=["date", "terms"])
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"])

        df["has_speaker"] = df["speaker"].notna()
        df["has_known_party"] = df["party"].notna() & df["party.facts.id"].notna()
        df["country"] = df["iso3country"].fillna("UNK")

        if "location" not in df.columns:
            df["location"] = None
        if "chamber" not in df.columns:
            df["chamber"] = None

        grouped = df.groupby(["country", df["date"].dt.date])

        for (country, date), group in grouped:
            meta_output.append({
                "country": country,
                "date": date,
                "location": group["location"].dropna().unique().tolist()[0] if group["location"].notna().any() else None,
                "chamber": group["chamber"].dropna().unique().tolist()[0] if group["chamber"].notna().any() else None,
                "total_words": group["terms"].sum(),
                "num_speeches": len(group),
                "with_speaker": group["has_speaker"].sum(),
                "with_known_speaker": group["has_known_party"].sum()
            })

    except Exception as e:
        print(f"⚠️ Failed on file {file_path}: {e}")

# --- Save and preview ---
meta_df = pd.DataFrame(meta_output)
meta_df = meta_df.sort_values(by=["country", "date"])
meta_df.to_csv(os.path.join(output_dir, "parlspeech_meta_daily.csv"), index=False)
meta_df.head()


,country,date,location,chamber,total_words,num_speeches,with_speaker,with_known_speaker
15293,CZE,1993-01-01,None,None,3969,8,8,7
15294,CZE,1993-01-26,None,None,25748,160,160,114
15295,CZE,1993-01-27,None,None,28099,245,245,144
15296,CZE,1993-01-28,None,None,38116,295,295,243
15297,CZE,1993-02-02,None,None,2697,15,15,14


In [19]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# === CONFIG ===
plot_dir = "/home/tom/projects/corpora/data/processed/country_reports_scaled"
output_html = "/home/tom/projects/corpora/data/processed/Combined_Parla_Report.html"
os.makedirs(plot_dir, exist_ok=True)

# Cutoff years for ParlaSpeech data per ParlaMint country code
cutoff_years = {
    "DE": 2022,
    "ES": 2015,
    "DK": 2014,
    "GB": 2015,
    "NZ": 2022,
    "CZ": 2014,
    "NL": 2014,
}

# Mapping from ParlaSpeech → ParlaMint
ps_to_pm = {
    "NLD": "NL",
    "ESP": "ES",
    "DNK": "DK",
    "GBR": "GB",
    "CZE": "CZ",
    "NZL": "NZ",
}

# Reverse mapping
pm_to_ps = {v: k for k, v in ps_to_pm.items()}

# === Load data ===
df_pm = pd.read_csv("/home/tom/projects/corpora/data/processed/parlamint_daily.csv")
df_ps = pd.read_csv("/home/tom/projects/corpora/data/processed/parlspeech_meta_daily.csv")

# Ensure datetime types
df_pm["date"] = pd.to_datetime(df_pm["date"], errors="coerce")
df_ps["date"] = pd.to_datetime(df_ps["date"], errors="coerce")

df_pm["month"] = df_pm["date"].dt.to_period("M").dt.to_timestamp()
df_ps["month"] = df_ps["date"].dt.to_period("M").dt.to_timestamp()

monthly_pm = df_pm.groupby(["country", "month"]).agg(total_words=("total_words", "sum")).reset_index()
monthly_ps = df_ps.groupby(["country", "month"]).agg(total_words=("total_words", "sum")).reset_index()

# === Helper ===
def extract_country_stats(pm_code, df_pm, monthly_pm, df_ps=None, monthly_ps=None, cutoff=None):
    stats = {"country": pm_code}

    pm_df = df_pm[df_pm["country"] == pm_code]
    pm_monthly = monthly_pm[monthly_pm["country"] == pm_code]

    ps_df = df_ps.copy() if df_ps is not None else pd.DataFrame()
    ps_monthly = monthly_ps.copy() if monthly_ps is not None else pd.DataFrame()

    if cutoff and not ps_df.empty:
        ps_df = ps_df[ps_df["date"] < datetime(cutoff, 1, 1)]
        ps_monthly = ps_monthly[ps_monthly["month"] < datetime(cutoff, 1, 1)]

    # Plot
    img_name = f"{pm_code}_combined_plot.png"
    img_path = os.path.join(plot_dir, img_name)
    plt.figure(figsize=(8, 3))
    if not ps_monthly.empty:
        sns.lineplot(data=ps_monthly, x="month", y="total_words", label="ParlaSpeech", color="blue")
    if not pm_monthly.empty:
        sns.lineplot(data=pm_monthly, x="month", y="total_words", label="ParlaMint", color="orange")
    plt.title(f"{pm_code} – Words per Month")
    plt.xlabel("Month")
    plt.ylabel("Words")
    plt.tight_layout()
    plt.savefig(img_path, dpi=100, bbox_inches="tight")
    plt.close()
    stats["image"] = img_path

    if not pm_df.empty:
        stats["pm"] = {
            "total_words": int(pm_df["total_words"].sum()),
            "total_speeches": int(pm_df["num_speeches"].sum()),
            "with_speaker": int(pm_df["with_speaker"].sum()),
            "with_known": int(pm_df["with_known_speaker"].sum()),
            "first_date": str(pm_df["date"].min().date()),
            "last_date": str(pm_df["date"].max().date())
        }

    if not ps_df.empty:
        stats["ps"] = {
            "total_words": int(ps_df["total_words"].sum()),
            "total_speeches": int(ps_df["num_speeches"].sum()),
            "with_speaker": int(ps_df["with_speaker"].sum()),
            "with_known": int(ps_df["with_known_speaker"].sum()),
            "first_date": str(ps_df["date"].min().date()),
            "last_date": str(ps_df["date"].max().date())
        }

    return stats

# === HTML Builder ===
def generate_html_report(stats_list, output_path):
    html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Combined ParlaMint + ParlaSpeech Report</title>
    <style>
        body { font-family: sans-serif; margin: 40px; }
        h2 { margin-top: 2em; border-bottom: 1px solid #ccc; padding-bottom: 0.2em; }
        h3 { margin-top: 1.5em; }
        img { max-width: 100%; height: auto; margin-top: 10px; }
        .stat { margin: 5px 0; }
    </style>
</head>
<body>
    <h1>📊 Combined Country-Level Report: ParlaMint & ParlaSpeech</h1>
"""

    for stats in stats_list:
        html += f"<h2>{stats['country']}</h2>"

        if "pm" in stats:
            pm = stats["pm"]
            html += f"""
            <h3>ParlaMint</h3>
            <div class='stat'><strong>Total words:</strong> {pm['total_words']:,}</div>
            <div class='stat'><strong>Total speeches:</strong> {pm['total_speeches']:,}</div>
            <div class='stat'><strong>Speeches with speaker ID:</strong> {pm['with_speaker']:,}</div>
            <div class='stat'><strong>With known speaker metadata:</strong> {pm['with_known']:,}</div>
            <div class='stat'><strong>Date range:</strong> {pm['first_date']} → {pm['last_date']}</div>
            """

        if "ps" in stats:
            ps = stats["ps"]
            cutoff_text = cutoff_years.get(stats["country"], "N/A")
            html += f"""
            <h3>ParlaSpeech (up to {cutoff_text})</h3>
            <div class='stat'><strong>Total words:</strong> {ps['total_words']:,}</div>
            <div class='stat'><strong>Total speeches:</strong> {ps['total_speeches']:,}</div>
            <div class='stat'><strong>Speeches with speaker ID:</strong> {ps['with_speaker']:,}</div>
            <div class='stat'><strong>With known speaker metadata:</strong> {ps['with_known']:,}</div>
            <div class='stat'><strong>Date range:</strong> {ps['first_date']} → {ps['last_date']}</div>
            """

        rel_path = os.path.relpath(stats["image"], os.path.dirname(output_path))
        html += f"<img src='{rel_path}' alt='{stats['country']} plot'/>"

    html += "\n</body></html>"

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"\u2705 Report saved to {output_path}")

# === Run ===
all_stats = []
for pm_code in sorted(df_pm["country"].unique()):
    ps_code = pm_to_ps.get(pm_code)
    cutoff = cutoff_years.get(pm_code)

    ps_df = df_ps[df_ps["country"] == ps_code] if ps_code else None
    ps_monthly = monthly_ps[monthly_ps["country"] == ps_code] if ps_code else None

    stats = extract_country_stats(
        pm_code, df_pm, monthly_pm,
        df_ps=ps_df,
        monthly_ps=ps_monthly,
        cutoff=cutoff
    )
    all_stats.append(stats)

generate_html_report(all_stats, output_html)

✅ Report saved to /home/tom/projects/corpora/data/processed/Combined_Parla_Report.html
